## Importing data and defining functions

In [110]:
import numpy as np
import pandas as pd
import tensorflow as tf  
from tensorflow.keras.models import Sequential 
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.regularizers import l1, l2, l1_l2
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, fbeta_score, precision_score, recall_score

X_test = pd.read_csv("Preprocessed_Data/X_test.csv")
y_test = pd.read_csv("Preprocessed_Data/y_test.csv").squeeze()
X_train = pd.read_csv("Preprocessed_Data/X_train.csv")
y_train = pd.read_csv("Preprocessed_Data/y_train.csv").squeeze()
X_valid = pd.read_csv("Preprocessed_Data/X_validation.csv")
y_valid = pd.read_csv("Preprocessed_Data/y_validation.csv").squeeze()

X_test_nn = X_test.to_numpy()
X_valid_nn = X_valid.to_numpy()
X_train_nn = X_train.to_numpy()

y_test_nn = y_test.to_numpy()
y_valid_nn = y_valid.to_numpy()
y_train_nn = y_train.to_numpy()

In [111]:
def create_model(l1_value=0, l2_value=0, dropout_rate=0):
    if l1_value > 0 and l2_value > 0:
        regularizer = l1_l2(l1=l1_value, l2=l2_value)
    elif l1_value > 0:
        regularizer = l1(l1_value)
    elif l2_value > 0:
        regularizer = l2(l2_value)
    else:
        regularizer = None
    model = Sequential([Input(shape=(X_train_nn.shape[1],)),Dense(16, activation="relu", kernel_regularizer=regularizer), Dense(8, activation="relu", kernel_regularizer=regularizer), Dropout(dropout_rate), Dense(1, activation="sigmoid")])
    model.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])
    return model

Firstly, all required data and libraries are imported. This time, the datasets are converted into NumPy arrays to make them easier to use with the neural network.
Then, a function for creating the model is defined. This function describes the structure of the future models by specifying their main settings, such as regularization and dropout, which help reduce overfitting.
The model consists of two hidden layers and one output layer. The first hidden layer has 16 neurons, and the second hidden layer has 8 neurons. Both hidden layers use the ReLU activation function. The last layer has one neuron and uses the sigmoid activation function, because the task is binary classification.

In [112]:
def train_and_evaluate_model(model, model_name):
    early_stop = EarlyStopping(monitor="val_loss",patience=10,restore_best_weights=True)

    history = model.fit(X_train_nn,y_train_nn,validation_data=(X_valid_nn, y_valid_nn),epochs = 100,batch_size = 32,callbacks=[early_stop],verbose=0)
    y_valid_proba = model.predict(X_valid_nn).ravel()
    y_valid_pred = (y_valid_proba >= 0.5).astype(int)
    accuracy = accuracy_score(y_valid_nn, y_valid_pred)
    precision = precision_score(y_valid_nn, y_valid_pred, zero_division=0)
    recall = recall_score(y_valid_nn, y_valid_pred, zero_division=0)
    f2 = fbeta_score(y_valid_nn, y_valid_pred, beta=2, zero_division=0)

    print(model_name)
    print("Validation accuracy:", accuracy)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F2-score:", f2)
    print("Confusion matrix:")
    print(confusion_matrix(y_valid_nn, y_valid_pred))
    print("Classification report:")
    print(classification_report(y_valid_nn, y_valid_pred, zero_division=0))

    return {"name": model_name,"model": model,"history": history,"accuracy": accuracy,"precision": precision,"recall": recall,"f2": f2}

Then a function for training and evaluating neural network models is created. This function trains the model on the training data and generates prediction results on the validation set. 
The model is trained  for a maximum of 100 epochs with a batch size of 32. But the training process can stop earlier if the validation loss does not become better for 10 epochs. At the end, the model keep the version that performed the best during training. 


## Creating models

In [113]:
results = []
models_to_train = [("Neural Network without regularization", 0, 0, 0), ("Neural Network with L1 regularization", 0.001, 0, 0), ("Neural Network with L2 regularization", 0, 0.001, 0), ("Neural Network with Dropout", 0, 0, 0.3), ("Neural Network with L1, L2 and Dropout", 0.0005, 0.0005, 0.3)
]
for name, l1_value, l2_value, dropout_rate in models_to_train:
    model = create_model(l1_value=l1_value, l2_value=l2_value, dropout_rate=dropout_rate)
    result = train_and_evaluate_model(model, name)
    results.append(result)

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
Neural Network without regularization
Validation accuracy: 0.8315217391304348
Precision: 0.8514851485148515
Recall: 0.8431372549019608
F2-score: 0.8447937131630648
Confusion matrix:
[[67 15]
 [16 86]]
Classification report:
              precision    recall  f1-score   support

           0       0.81      0.82      0.81        82
           1       0.85      0.84      0.85       102

    accuracy                           0.83       184
   macro avg       0.83      0.83      0.83       184
weighted avg       0.83      0.83      0.83       184

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
Neural Network with L1 regularization
Validation accuracy: 0.8641304347826086
Precision: 0.8888888888888888
Recall: 0.8627450980392157
F2-score: 0.8678500986193294
Confusion matrix:
[[71 11]
 [14 88]]
Classification report:
              precision    recall  f1-score   support

           0       0.84      0.87      0.85        82
           1       0.89      0.86      0

Five neural network models were created and compared. The first model was trained without regularization, while the following models used different techniques that helped to reduce overfitting, such as L1 regularization, L2 regularization, drop out, and a combination of these methods.

## Choosing the best model

In [114]:
best_result = max(results, key=lambda x: x["f2"])
best_nn_model = best_result["model"]
print("Best neural network model:", best_result["name"])
print("Best validation F2:", best_result["f2"])
print("Validation recall:", best_result["recall"])
print("Validation precision:", best_result["precision"])
print("Validation accuracy:", best_result["accuracy"])

Best neural network model: Neural Network with L1 regularization
Best validation F2: 0.8678500986193294
Validation recall: 0.8627450980392157
Validation precision: 0.8888888888888888
Validation accuracy: 0.8641304347826086


After all models were trained, the F2-score was used to choose the best of them in terms of recall priority. The code compares all saved model results and selects the one with the highest F2-score.
As a result, the best model was the neural network with L1 regularization. It had a recall of 86.27%, precision of 88.89%, and accuracy of 86.41%. 


## First prediction 

In [115]:
y_valid_proba_nn = best_nn_model.predict(X_valid_nn).ravel()
thresholds = np.arange(0.1, 1, 0.01)
scores = []
for threshold in thresholds:
    y_valid_thresh = (y_valid_proba_nn >= threshold).astype(int)
    f2 = fbeta_score(y_valid_nn, y_valid_thresh, beta=2, zero_division=0)
    recall = recall_score(y_valid_nn, y_valid_thresh, zero_division=0)
    precision = precision_score(y_valid_nn, y_valid_thresh, zero_division=0)
    scores.append((threshold, f2, recall, precision))
best_nn_thresh, best_nn_f2, best_nn_recall, best_nn_precision = max(
    scores,
    key=lambda x: x[1]
)
print("Best neural network threshold:", round(best_nn_thresh, 2))
print("Best neural network F2:", best_nn_f2)
print("Recall:", best_nn_recall)
print("Precision:", best_nn_precision)

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
Best neural network threshold: 0.12
Best neural network F2: 0.923217550274223
Recall: 0.9901960784313726
Precision: 0.7266187050359713


The best threshold for the model was chosen using the validation set. After checking several thresholds, the 0.12 threshold was chosen. It gave a 99% recall, which means the model was able to detect almost all actual heart disease. 
But at the same time, precision decreased to 73%, which means that more false positive predictions were made. 


In [ ]:
y_valid_pred_nn_thresh = (y_valid_proba_nn >= best_nn_thresh).astype(int)
print("Neural Network validation accuracy with tuned threshold:", accuracy_score(y_valid_nn, y_valid_pred_nn_thresh))
print("Confusion matrix:")
print(confusion_matrix(y_valid_nn, y_valid_pred_nn_thresh))
print("Classification report:")
print(classification_report(y_valid_nn, y_valid_pred_nn_thresh, zero_division=0))

Neural Network validation accuracy with tuned threshold: 0.7880434782608695
Confusion matrix:
[[ 44  38]
 [  1 101]]
Classification report:
              precision    recall  f1-score   support

           0       0.98      0.54      0.69        82
           1       0.73      0.99      0.84       102

    accuracy                           0.79       184
   macro avg       0.85      0.76      0.77       184
weighted avg       0.84      0.79      0.77       184



The final validation prediction was made using the threshold found earlier. The model had an accuracy equal to 78.80%. The recall remained at 99%, so almost all actual heart disease cases were identified correctly. 
As the precision is 73%, some healthy patients were incorrectly classified as patients with heart disease. But it is worth it, as it makes the model miss less cases with actual heart disease. 


## Testing

In [117]:
y_test_proba_nn = best_nn_model.predict(X_test_nn).ravel()
y_test_pred_nn_thresh = (y_test_proba_nn >= best_nn_thresh).astype(int)
print("Neural Network test accuracy with tuned threshold:", accuracy_score(y_test_nn, y_test_pred_nn_thresh))
print("Best neural network model:", best_result["name"])
print("Best neural network threshold:", round(best_nn_thresh, 2))
print("Confusion matrix:")
print(confusion_matrix(y_test_nn, y_test_pred_nn_thresh))
print("Classification report:")
print(classification_report(y_test_nn, y_test_pred_nn_thresh, zero_division=0))

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
Neural Network test accuracy with tuned threshold: 0.7445652173913043
Best neural network model: Neural Network with L1 regularization
Best neural network threshold: 0.12
Confusion matrix:
[[42 40]
 [ 7 95]]
Classification report:
              precision    recall  f1-score   support

           0       0.86      0.51      0.64        82
           1       0.70      0.93      0.80       102

    accuracy                           0.74       184
   macro avg       0.78      0.72      0.72       184
weighted avg       0.77      0.74      0.73       184



Finally, the model was tested on the testing data using the chosen regularization model and a new threshold. 
The model achieved the accuracy of 74.46%, the recall of 93%, and the precision of 70%.
Although the model made some false positive predictions, it still correctly identifies most patients with heart disease, which was the main goal. 
